# Notebook 5 - Optimisation avec dataset enrichi

Objectif : comparer proprement le projet **avant** et **apres** l'ajout du dataset MyAnimeList `archive (1)`, avec une taxonomie regroupee et des seuils multilabel optimises.

La comparaison garde le dataset NovelForge actuel comme domaine de test afin de mesurer si l'enrichissement aide vraiment le probleme initial.

## Strategie

- Regrouper les labels rares ou ambigus apres analyse des volumes.
- Fusionner `Hentai`, `Ecchi`, `Erotica`, `Smut` en `Adult`.
- Fusionner `BL`, `GL`, `Yaoi`, `Yuri`, `Boys Love`, `Girls Love`, `Shounen-ai`, `Shoujo-ai` en `BL/GL Romance`.
- Utiliser `manga_dataset.csv` comme enrichissement principal, plus proche du domaine que l'anime.
- Comparer une baseline actuelle contre une baseline enrichie sur le meme test set NovelForge.

In [1]:
from pathlib import Path
import sys

# Works whether Jupyter starts from the project root or from notebooks/.
PROJECT_DIR = Path.cwd().resolve()
if PROJECT_DIR.name == 'notebooks':
    PROJECT_DIR = PROJECT_DIR.parent
elif not (PROJECT_DIR / 'src').exists() and (PROJECT_DIR.parent / 'src').exists():
    PROJECT_DIR = PROJECT_DIR.parent

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

import joblib
import pandas as pd
from sklearn.metrics import classification_report, f1_score, hamming_loss, jaccard_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer

from src.baseline_ml import (
    BaselineModel,
    apply_thresholds,
    find_best_global_threshold,
    find_best_label_thresholds,
)
from src.enriched_dataset import build_enriched_dataset, load_original_dataset, summarize_labels
from src.project_config import ENRICHED_GENRE_VOCABULARY

pd.set_option('display.max_colwidth', 120)
PROJECT_DIR

WindowsPath('C:/Users/ClémentPERRET/OneDrive - EQUATERRE-VDS/Bureau/Cours/DeepLearning')

In [2]:
current = load_original_dataset(PROJECT_DIR / 'data' / 'data.csv')
mal_manga = build_enriched_dataset(PROJECT_DIR, include_original=False, include_manga=True, include_anime=False)

print(f'Dataset actuel groupe : {len(current):,} lignes')
print(f'MAL manga exploitable : {len(mal_manga):,} lignes')

display(current.head(3))
display(mal_manga.head(3))

Dataset actuel groupe : 69,588 lignes
MAL manga exploitable : 56,587 lignes


,title,synopsis_clean,genre_labels,source_dataset,media_type
0,Salad Days (Tang LiuZang) - Part 2,the second season of salad day tang liuzang,"[BL/GL Romance, Romance, Sports]",novelforge_current,manga_lightnovel
1,The Master of Diabolism,as the grandmaster who found the demonic sect wei wuxian roam the world in his wanton way hat by million for the cha...,"[Action, Adventure, BL/GL Romance, Comedy, Mystery, Romance, Martial Arts, Supernatural, Isekai]",novelforge_current,manga_lightnovel
2,JoJo's Bizarre Adventure Part 7: Steel Ball Run,set in steel ball run spotlight gyro zepelli and johnny joestar as they pit their spirit on a fifty million dollar r...,"[Action, Adventure, Horror, Mystery, Seinen, Historical]",novelforge_current,manga_lightnovel


,title,synopsis_clean,genre_labels,source_dataset,media_type,label_count
0,Monster,kenzou tenma a renown japanese neurosurgeon work in post-war germany face a difficult choice to operate on johan lie...,"[Drama, Mystery, Psychological, Seinen]",manga_dataset,manga,4
1,Berserk,gut a former mercenary now known as the black swordsman is out for revenge after a tumultuou childhood he finally fi...,"[Action, Adventure, Drama, Fantasy, Horror, Psychological, Seinen]",manga_dataset,manga,7
2,20th Century Boys,as the century approache its end people all over the world are anxiou that the world is chang and probably not for t...,"[Drama, Mystery, Sci Fi, Historical, Psychological, Seinen]",manga_dataset,manga,6


In [3]:
current_counts = summarize_labels(current)
mal_counts = summarize_labels(mal_manga)

comparison_counts = current_counts.merge(mal_counts, on='label', suffixes=('_current', '_mal_manga'))
comparison_counts['total'] = comparison_counts['count_current'] + comparison_counts['count_mal_manga']
comparison_counts.sort_values('total', ascending=False).head(26)

,label,count_current,count_mal_manga,total
0,Romance,29762,14881,44643
1,Comedy,21282,11665,32947
2,Drama,18702,9478,28180
3,Fantasy,16125,10987,27112
4,BL/GL Romance,14659,12045,26704
14,Adult,5139,19214,24353
6,School Life,12582,9073,21655
5,Action,12734,8045,20779
8,Supernatural,8689,7035,15724
7,Seinen,9235,6480,15715


In [4]:
current_train, current_temp = train_test_split(current, test_size=0.30, random_state=42, shuffle=True)
current_valid, current_test = train_test_split(current_temp, test_size=0.50, random_state=42, shuffle=True)

after_train = pd.concat([current_train, mal_manga], ignore_index=True)

print(f'Train avant : {len(current_train):,}')
print(f'Validation : {len(current_valid):,}')
print(f'Test NovelForge : {len(current_test):,}')
print(f'Train apres : {len(after_train):,}')

Train avant : 48,711
Validation : 10,438
Test NovelForge : 10,439
Train apres : 105,298


In [5]:
mlb = MultiLabelBinarizer(classes=ENRICHED_GENRE_VOCABULARY)
mlb.fit([ENRICHED_GENRE_VOCABULARY])
labels = list(mlb.classes_)

y_current_train = mlb.transform(current_train['genre_labels'])
y_after_train = mlb.transform(after_train['genre_labels'])
y_valid = mlb.transform(current_valid['genre_labels'])
y_test = mlb.transform(current_test['genre_labels'])

print(f'Nombre de labels enrichis : {len(labels)}')
labels

Nombre de labels enrichis : 26


['Action',
 'Adventure',
 'Comedy',
 'Drama',
 'Fantasy',
 'Romance',
 'BL/GL Romance',
 'Adult',
 'School Life',
 'Slice of Life',
 'Supernatural',
 'Mystery',
 'Psychological',
 'Horror',
 'Historical',
 'Sci Fi',
 'Sports',
 'Martial Arts',
 'Magic',
 'Isekai',
 'Harem',
 'Mecha',
 'Seinen',
 'Shoujo',
 'Shounen',
 'Josei']

In [6]:
def train_model(x_train, y_train):
    model = BaselineModel(
        max_features=60_000,
        ngram_range=(1, 2),
        min_df=3,
        max_df=0.92,
        C=1.5,
        solver='lbfgs',
        max_iter=1_000,
    )
    return model.train(x_train, y_train)


def evaluate_from_probabilities(y_true, probabilities, thresholds):
    y_pred = apply_thresholds(probabilities, thresholds)
    return {
        'f1_micro': f1_score(y_true, y_pred, average='micro', zero_division=0),
        'f1_macro': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'f1_weighted': f1_score(y_true, y_pred, average='weighted', zero_division=0),
        'jaccard_samples': jaccard_score(y_true, y_pred, average='samples', zero_division=0),
        'hamming_loss': hamming_loss(y_true, y_pred),
        'classification_report_text': classification_report(y_true, y_pred, target_names=labels, zero_division=0),
        'classification_report': classification_report(y_true, y_pred, target_names=labels, zero_division=0, output_dict=True),
    }

In [7]:
before_model = train_model(current_train['synopsis_clean'], y_current_train)

before_valid_proba = before_model.predict_proba(current_valid['synopsis_clean'])
before_test_proba = before_model.predict_proba(current_test['synopsis_clean'])
before_threshold, before_valid_f1 = find_best_global_threshold(y_valid, before_valid_proba)
before_metrics = evaluate_from_probabilities(y_test, before_test_proba, before_threshold)

print(f'Seuil global avant : {before_threshold:.2f}')
print(f'F1 micro validation avant : {before_valid_f1:.4f}')
print(f'F1 micro test avant : {before_metrics["f1_micro"]:.4f}')

Seuil global avant : 0.55
F1 micro validation avant : 0.4803
F1 micro test avant : 0.4847


In [8]:
after_model = train_model(after_train['synopsis_clean'], y_after_train)

after_valid_proba = after_model.predict_proba(current_valid['synopsis_clean'])
after_test_proba = after_model.predict_proba(current_test['synopsis_clean'])
after_global_threshold, after_valid_f1 = find_best_global_threshold(y_valid, after_valid_proba)
after_label_thresholds = find_best_label_thresholds(y_valid, after_valid_proba)

after_global_metrics = evaluate_from_probabilities(y_test, after_test_proba, after_global_threshold)
after_label_metrics = evaluate_from_probabilities(y_test, after_test_proba, after_label_thresholds)

print(f'Seuil global apres : {after_global_threshold:.2f}')
print(f'F1 micro validation apres : {after_valid_f1:.4f}')
print(f'F1 micro test apres global : {after_global_metrics["f1_micro"]:.4f}')
print(f'F1 micro test apres seuils par label : {after_label_metrics["f1_micro"]:.4f}')

Seuil global apres : 0.60
F1 micro validation apres : 0.5078
F1 micro test apres global : 0.5144
F1 micro test apres seuils par label : 0.5379


In [9]:
metrics_summary = pd.DataFrame([
    {
        'model': 'before_current_only',
        'train_rows': len(current_train),
        'external_rows': 0,
        'threshold_strategy': 'global',
        'threshold': before_threshold,
        'valid_f1_micro': before_valid_f1,
        **{key: before_metrics[key] for key in ['f1_micro', 'f1_macro', 'f1_weighted', 'jaccard_samples', 'hamming_loss']},
    },
    {
        'model': 'after_current_plus_manga',
        'train_rows': len(after_train),
        'external_rows': len(mal_manga),
        'threshold_strategy': 'global',
        'threshold': after_global_threshold,
        'valid_f1_micro': after_valid_f1,
        **{key: after_global_metrics[key] for key in ['f1_micro', 'f1_macro', 'f1_weighted', 'jaccard_samples', 'hamming_loss']},
    },
    {
        'model': 'after_current_plus_manga',
        'train_rows': len(after_train),
        'external_rows': len(mal_manga),
        'threshold_strategy': 'per_label',
        'threshold': None,
        'valid_f1_micro': None,
        **{key: after_label_metrics[key] for key in ['f1_micro', 'f1_macro', 'f1_weighted', 'jaccard_samples', 'hamming_loss']},
    },
])
metrics_summary

,model,train_rows,external_rows,threshold_strategy,threshold,valid_f1_micro,f1_micro,f1_macro,f1_weighted,jaccard_samples,hamming_loss
0,before_current_only,48711,0,global,0.55,0.480251,0.484729,0.438804,0.487236,0.287025,0.113815
1,after_current_plus_manga,105298,56587,global,0.60,0.507829,0.514441,0.481414,0.516846,0.310659,0.103130
2,after_current_plus_manga,105298,56587,per_label,NaN,NaN,0.537885,0.491304,0.534920,0.372231,0.119006


In [10]:
report_after = pd.DataFrame(after_label_metrics['classification_report']).T
report_after.loc[labels, ['precision', 'recall', 'f1-score', 'support']].sort_values('f1-score', ascending=False)

,precision,recall,f1-score,support
Romance,0.520935,0.895958,0.658816,4527.0
Isekai,0.633094,0.680412,0.655901,388.0
BL/GL Romance,0.696311,0.619326,0.655566,2225.0
Fantasy,0.752669,0.521150,0.615870,2435.0
School Life,0.640240,0.544990,0.588788,1956.0
Action,0.655541,0.531098,0.586794,1849.0
Sports,0.723404,0.451327,0.555858,226.0
Comedy,0.394585,0.855015,0.539975,3290.0
Josei,0.647788,0.462705,0.539823,791.0
Mecha,0.612903,0.452381,0.520548,84.0


In [11]:
MODELS_DIR = PROJECT_DIR / 'models'
REPORTS_DIR = PROJECT_DIR / 'reports'
MODELS_DIR.mkdir(exist_ok=True)
REPORTS_DIR.mkdir(exist_ok=True)

joblib.dump(after_model, MODELS_DIR / 'enhanced_tfidf.joblib')
joblib.dump(labels, MODELS_DIR / 'enhanced_labels.joblib')
joblib.dump(after_label_thresholds, MODELS_DIR / 'enhanced_thresholds.joblib')
joblib.dump(
    {
        'before': before_metrics,
        'after_global': after_global_metrics,
        'after_per_label': after_label_metrics,
        'summary': metrics_summary,
        'labels': labels,
        'before_threshold': before_threshold,
        'after_global_threshold': after_global_threshold,
        'after_label_thresholds': after_label_thresholds,
    },
    MODELS_DIR / 'enhanced_metrics.joblib',
)
metrics_summary.to_csv(REPORTS_DIR / 'enhanced_before_after_metrics.csv', index=False)
comparison_counts.to_csv(REPORTS_DIR / 'enriched_label_count_comparison.csv', index=False)

print('Artefacts sauvegardes dans models/')
print('Rapports sauvegardes dans reports/')

Artefacts sauvegardes dans models/
Rapports sauvegardes dans reports/


## Conclusion

L'ajout du dataset MyAnimeList `manga_dataset.csv` ameliore bien le modele sur le domaine de test NovelForge. La comparaison reste propre, car les scores sont calcules sur le meme test set issu du dataset initial.

Resultats observes :

- baseline avant, entrainee seulement sur le dataset courant : F1 micro `0.485`, F1 macro `0.439`, Jaccard samples `0.287` ;
- baseline apres, entrainee sur le dataset courant + MAL manga avec seuil global : F1 micro `0.514`, F1 macro `0.481`, Jaccard samples `0.311` ;
- baseline apres avec seuils optimises par label : F1 micro `0.538`, F1 macro `0.491`, Jaccard samples `0.372`.

Le meilleur compromis pour l'interface est donc la baseline enrichie avec seuils par label : elle exploite plus de donnees proches du domaine manga/manhwa et beneficie d'une taxonomie plus stable. La Hamming loss augmente legerement avec les seuils par label (`0.119` contre `0.103` avec seuil global), ce qui indique que le modele predit davantage de labels positifs. En contrepartie, il recouvre mieux les genres pertinents, ce qui est plus adapte a un assistant d'annotation multilabel.

Cette experience confirme que l'amelioration vient moins d'un modele plus complexe que d'un meilleur travail sur les donnees : enrichissement du corpus, regroupement des labels bruyants et calibration des seuils.